# CEADS parser walkthrough

This notebook is the practical guide for parsing the currently supported CEADS China provincial MRIO workbooks in MARIO.

## What this notebook covers

- where the supported CEADS workbooks come from;
- which workbook family MARIO currently parses;
- what `format=` means;
- how to use `path=` and `year=`;
- how MARIO maps imports, exports, and the optional `CO2` row.

## Relevant source pages

- CEADS portal: [Input-Output Tables](https://www.ceads.net/data/input_output_tables/)
- Scientific Data descriptor: [China’s Provincial Multi-Regional Input-Output Database for 2018 and 2020](https://www.nature.com/articles/s41597-025-06543-y)
- Dataset DOI: [10.6084/m9.figshare.29927291](https://doi.org/10.6084/m9.figshare.29927291)

MARIO does not download the CEADS workbook automatically. The expected workflow is to download one workbook yourself and then parse it locally.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_ceads(...)`

At the moment, that entry point supports only `table="IOT"`.

## What MARIO currently parses

The current backend is intentionally narrow:

- it parses the local Excel workbook only;
- it currently supports the verified CEADS provincial MRIO workbook family for `2018` and `2020`;
- it reads the English table sheet, not the Chinese one;
- it uses the `Sector` and `Province` sheets as metadata support.

## What `format` means

`format=` is the parser-side layout selector. It does not mean "which CEADS dataset" you are using; it means "which workbook structure" MARIO should expect.

At the moment, MARIO supports:

- `format="auto"`: inspect the workbook and resolve the known layout automatically;
- `format="ceads_provincial_workbook"`: force the verified 2018/2020 workbook structure.

## How the parsed table is mapped

For the verified CEADS workbook family:

- `Z` comes from the main intermediate-transactions block;
- `Y` contains the domestic final-demand block plus one `Exports` column per province;
- `V` contains `Imports` plus the four value-added rows exposed by the workbook;
- `E` contains the optional `CO2 (Mt)` row when it exists;
- `EY` is initialized at zero.

## Typical usage

Parse one explicit workbook:

```python
import mario

db = mario.parse_ceads(
    path="/path/to/MRIO 2018.xlsx",
    format="auto",
)
```

Parse one directory containing multiple CEADS workbooks:

```python
import mario

db = mario.parse_ceads(
    path="/path/to/ceads_directory",
    year=2020,
)
```

## Caveats

- The CEADS portal lists older provincial MRIO releases too, such as `2012`, `2015`, and `2017`. MARIO does not assume those older workbook families are compatible until they are checked explicitly.
- The parser intentionally relies on the English worksheet. If only a Chinese worksheet is available, that workbook is not currently supported.
- The workbook metadata sheets can contain minor naming inconsistencies. For the verified 2018/2020 family, MARIO treats the main transaction axis as canonical when such conflicts appear.